In [1]:
from vab_rt_import.source import get_data
df = get_data()

In [2]:
df

,id,cognome_socio,nome_socio,num_tessera,num_tessera_associazione,lettera_id_regione_vab,flag_socio,flag_volontario,socio_inattivo,qualifica_socio,...,flag_stampa_ok,flag_stampa_dataora,flag_stampa_id_op,auth_lavori_sede,porta,attivazione_rt_2020,attivazione_locale_2020,da_vaccinare_covid19,data_cambio_password,blocca_account_cambio_password
0,5,Polverini,Andreao,8248,0,T,on,on,0,2,...,,0000-00-00 00:00:00,0,,on,,,,2026-01-03 10:54:32,0
1,7,Magazzini,Nicola,5677,0,T,on,on,1,1,...,,0000-00-00 00:00:00,0,,,,,,0000-00-00 00:00:00,0
2,8,Olivari,Pier Riccardo,6210,0,T,on,on,1,1,...,,0000-00-00 00:00:00,0,,,0,0,0,0000-00-00 00:00:00,0
3,11,Valdettari,Riccardo,7614,0,T,on,on,1,3,...,,0000-00-00 00:00:00,0,,,,,,0000-00-00 00:00:00,0
4,13,Burroni,Maurizio,7389,0,T,on,on,0,1,...,,0000-00-00 00:00:00,0,,,,,,0000-00-00 00:00:00,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9326,9698,Curella,Matteo,,0,T,,,0,2,...,,0000-00-00 00:00:00,0,,,,,,0000-00-00 00:00:00,0
9327,9699,Bianchi,Andrea,,0,T,on,on,0,1,...,,0000-00-00 00:00:00,0,on,,,,,0000-00-00 00:00:00,0
9328,9700,Ugolini,Dimitri,,0,T,,,0,1,...,,0000-00-00 00:00:00,0,,,,,,0000-00-00 00:00:00,0
9329,9701,Capocaccia,Matteo,,0,T,,on,0,9,...,,0000-00-00 00:00:00,0,,,,,,0000-00-00 00:00:00,0


## Data quality

In [3]:
import pandas as pd
from vab_rt_import.utils.utils import data_quality_report, is_empty

In [4]:
data_quality_report(df)

,missing,missing_%,empty_str,unique,dtype,present,present_%
scadenza_doc_identita_socio,8750,93.77,0,17,object,581,6.23
cert_medico,7274,77.96,0,12,object,2057,22.04
patente_numero,885,9.48,5671,2692,str,8446,90.52
brevetti,885,9.48,7184,777,str,8446,90.52
titstudio,885,9.48,5878,794,str,8446,90.52
...,...,...,...,...,...,...,...
flag_attivita_protciv,0,0.00,6085,2,str,9331,100.00
flag_attivita_antincendio,0,0.00,6679,2,str,9331,100.00
email,0,0.00,3370,4576,str,9331,100.00
tel_altro,0,0.00,7671,314,str,9331,100.00


## Split table into groups

In order to analyze better the data, we should try to create group of one or more groups, which are connected  by `id` field, since it is verified that it is a pk.

In [5]:
list(df.columns)

['id',
 'cognome_socio',
 'nome_socio',
 'num_tessera',
 'num_tessera_associazione',
 'lettera_id_regione_vab',
 'flag_socio',
 'flag_volontario',
 'socio_inattivo',
 'qualifica_socio',
 'data_disattivazione_socio',
 'note_disattivazione_socio',
 'carica_socio',
 'data_iscriz',
 'data_registr_libro_soci',
 'rinnovo_tessera',
 'nato_a',
 'nato_il',
 'cod_fisc',
 'sesso_socio',
 'id_sesso_socio',
 'residente_a',
 'via',
 'num_civico',
 'cap',
 'provincia',
 'tipo_doc_identita_socio',
 'numero_doc_identita_socio',
 'scadenza_doc_identita_socio',
 'doc_identita_socio_rilasc_da',
 'tel_abit',
 'tel_cellulare',
 'tel_altro',
 'email',
 'flag_attivita_antincendio',
 'flag_attivita_protciv',
 'flag_attivita_ecologiche',
 'flag_attivita_logistiche',
 'flag_attivita_altro',
 'nome_azienda',
 'indirizzo_azienda',
 'cap_azienda',
 'comune_azienda',
 'prov_azienda',
 'tel_azienda',
 'fax_azienda',
 'email_azienda',
 'pec_azienda',
 'patente_tipo',
 'patente_numero',
 'pat_rilasc_il',
 'pat_rilasc_d

# Data cleaning

Start the cleaning of data with fiscal data:

* Name
* Surname
* Gender
* Birthplace
* Birthdate

In [6]:
GROUPS = {
    "FISCAL": [
        "nome_socio",
        "cognome_socio",
        "nato_a",
        "nato_il",
        "cod_fisc",
        "sesso_socio",
        "id_sesso_socio",
    ],
    "SUBSCRIPTIONS": [
        "num_tessera",
        "num_tessera_associazione",
        "lettera_id_regione_vab",
        "flag_socio",
        "flag_volontario",
        "socio_inattivo",
        "data_disattivazione_socio",
        "note_disattivazione_socio",
        "data_registr_libro_soci",
        "rinnovo_tessera",
    ],
    "DOCS": [
        "tipo_doc_identita_socio",
        "numero_doc_identita_socio",
        "scadenza_doc_identita_socio",
        "doc_identita_socio_rilasc_da",
        "patente_tipo",
        "patente_numero",
        "pat_rilasc_il",
        "pat_rilasc_da",
        "pat_scad",
    ],
    "CONTACTS": [
        "residente_a",
        "via",
        "num_civico",
        "cap",
        "provincia",
        "tel_abit",
        "tel_cellulare",
        "tel_altro",
        "email",
    ],
    "MEMBERSHIPS": [
        "id_sezione_vab",
    ],
    "WORK": [
        "nome_azienda",
        "indirizzo_azienda",
        "cap_azienda",
        "comune_azienda",
        "prov_azienda",
        "tel_azienda",
        "fax_azienda",
        "email_azienda",
        "pec_azienda",
        "caratt_lavorative",
        "altra_formazione_esterna",
    ],
    "HEALTH": [
        'gruppo_sanguigno',
        'scad_antitetanica',
        'antitetanica',
        'allergie_e_patologie',
        'donatore',
        'autocertif',
        'cert_medico',
        'scadenza_visita_medica',
        'idoneita_medica_socio',
        'professione',
    ],
    "DPI": [
        "taglia_piede",
        "taglia_vest",
        "taglia_guanti",
        "taglia_elmetto",
    ],
    "PERMISSIONS": [
        'access_anagr_globali',
         'access_interv',
         'access_amm',
         'access_anagr_mezzi',
         'access_anagr_soci',
         'access_anagr_varie',
         'access_totale',
         'access_sanitario',
         'access_assicurazioni',
         'access_formazione',
         'access_bacheca_sez_emergenze',
         'access_bacheca_sez_servizi',
         'abilitazione_socio_segreteria',
         'abilitazione_socio_presidente',
         'abilitazione_socio_presidente_dataora',
         'abilitazione_socio_presidente_id_op',
         'access_stampa_schede_iscrizione',
    ]
}

In [13]:
total = [field for fields in GROUPS.values() for field in fields]
set(df.columns).difference(set(total))

{'access_time',
 'app_altre_ass',
 'attivazione_locale_2020',
 'attivazione_rt_2020',
 'auth_lavori_sede',
 'badge',
 'badge2',
 'blocca_account_cambio_password',
 'brevetti',
 'carica_socio',
 'conc_radio',
 'da_vaccinare_covid19',
 'data_cambio_password',
 'data_iscriz',
 'data_operatore_ins',
 'data_operatore_variaz',
 'flag_attivita_altro',
 'flag_attivita_antincendio',
 'flag_attivita_ecologiche',
 'flag_attivita_logistiche',
 'flag_attivita_protciv',
 'flag_stampa_dataora',
 'flag_stampa_id_op',
 'flag_stampa_ok',
 'hobby_sport',
 'id',
 'id_ingresso_porta_sede',
 'id_operatore_ins',
 'id_operatore_variaz',
 'note',
 'num_tessera_socio_presentatore',
 'password',
 'password_mai_cambiata',
 'porta',
 'qualifica_socio',
 'rif_foto',
 'sigla_radio',
 'socio_presentatore',
 'tessera_pc_olma',
 'titstudio',
 'try'}

In [ ]:
def clean_name(name):
    if name is None or (isinstance(name, float) and pd.isna(name)):
        return pd.NA

    name = normalize_txt(name)
    name = name.strip()
    name = re.sub(r"\s+", " ", name)
    name = re.sub(r"[^A-Za-zÀ-ÖØ-öø-ÿ' -]", "", name)
    name = name.lower()

    def smart_cap(word):
        if "'" in word:
            return "'".join(w.capitalize() for w in word.split("'"))
        if "-" in word:
            return "-".join(w.capitalize() for w in word.split("-"))
        return word.capitalize()

    name = " ".join(smart_cap(w) for w in name.split())

    name = name if len(name) > 1 else pd.NA

    return name

In [ ]:
df['cognome_socio'] = df['cognome_socio'].apply(clean_name)

In [ ]:
df['nome_socio'] = df['nome_socio'].apply(clean_name)

### Rows without name or surname must be dropped.

In [ ]:
INVALID_NAMES = [
    'test',
    'prova',
    'paperopoli',
    'xxx',
]
NAMES_COLUMNS = ['nome_socio', 'cognome_socio']

In [ ]:
def clean_not_recoverable_rows(row):
    drop = False
    for c in NAMES_COLUMNS:
        if drop:
            break
        elif pd.isna(row[c]):
            drop = True
        else:
            drop = row[c].lower().strip() in INVALID_NAMES

    row['_drop'] = drop
    return row

In [ ]:
_drops = df.apply(clean_not_recoverable_rows, axis=1)
df_dropped = df[_drops['_drop']]

In [ ]:
_drops['_drop'].value_counts()

In [ ]:
_drops[_drops['_drop'] == True]

In [ ]:
df = df[~_drops['_drop']]
del _drops

In [ ]:
df['sesso_socio'].value_counts(dropna=False)

In [ ]:
df['id_sesso_socio'].value_counts(dropna=False)

In [ ]:
df[df['sesso_socio'] == 'm']['id_sesso_socio'].value_counts().plot.bar()

In [ ]:
df[df['sesso_socio'] == 'f']['id_sesso_socio'].value_counts().plot.bar()

In [ ]:
df[['id_sesso_socio', 'sesso_socio']].value_counts().plot.bar()

Uhm, `sesso_socio` and `id_sesso_socio` are dirty, we need to use probability to
infer gender.

We need to compute:
$$
P(M | ID=k) , P(F | ID=k) = P(\text{sesso\_socio} = S | \text{id\_sesso\_socio}=k)
$$


In [ ]:
# count all combinations
counts = df.groupby(['id_sesso_socio', 'sesso_socio']).size().unstack(fill_value=0)
# compute prob
prob = counts.div(counts.sum(axis=1), axis=0)
# transform in a dict p_g_id
p_g_id = {g: prob[g].to_dict() for g in prob.columns}

p_g_id

In [ ]:
def clean_gender(gender):
    if not gender or pd.isna(gender):
        return pd.NA

    gender = str(gender).strip().lower()

    if gender.startswith('m'):
        return 'm'
    if gender.startswith('f'):
        return 'f'

    return pd.NA


In [ ]:
df['sesso_socio'].apply(clean_gender).value_counts(dropna=False).plot.bar()

In [ ]:
df['sesso_socio'] = df['sesso_socio'].apply(clean_gender)

In [ ]:
def resolve_gender(row, threshold=0.82):
    if pd.notna(row['sesso_socio']) and row['sesso_socio'] in ['m', 'f']:
        return row['sesso_socio']

    _id = row['id_sesso_socio']

    p_m = p_g_id.get('m', {}).get(_id, 0)
    p_f = p_g_id.get('f', {}).get(_id, 0)

    total = p_m + p_f

    if total == 0:
        return pd.NA

    p_m /= total
    p_f /= total

    if p_m >= threshold:
        return 'm'
    elif p_f >= threshold:
        return 'f'
    else:
        return pd.NA

In [ ]:
(df[['sesso_socio', 'id_sesso_socio']].apply(resolve_gender, axis=1).value_counts
 (dropna=False))

In [ ]:
df['sesso_socio'] = df[['sesso_socio', 'id_sesso_socio']].apply(resolve_gender, axis=1)

In [ ]:
df[df['sesso_socio'].isna()][['id', 'nome_socio', 'cognome_socio', 'sesso_socio',
                              'id_sesso_socio', 'cod_fisc', 'nato_il']]

In [ ]:
# drop is_sesso_socio
df = df.drop(columns=['id_sesso_socio'])

In [ ]:
def clean_date(date):
    if not date or pd.isna(date):
        return pd.NaT
    return pd.to_datetime(date, errors='coerce', format="%d/%m/%Y")

In [ ]:
df['nato_il'] = df['nato_il'].apply(clean_date)

In [ ]:
df[['nome_socio', 'cognome_socio', 'nato_il']]

In [ ]:
def clean_cf(cf):
    if is_empty(cf):
        return pd.NA
    r = str(cf).strip().upper()
    if len(r) not in (15, 16):
        return pd.NA
    return r

In [ ]:
df['cod_fisc'].apply(clean_cf).isna().value_counts()

In [ ]:
df['cod_fisc'] = df['cod_fisc'].apply(clean_cf)

In [ ]:
def clean_birthplace(place):
    if not place or pd.isna(place):
        return pd.NA
    if len(place) <= 1:
        return pd.NA

    place = str(place).strip()
    place = re.sub(r"\s+", " ", place)

    place = re.sub(r"\s*\(\s*", " (", place)
    place = re.sub(r"\s*\)\s*", ")", place)

    parts = place.split(" ")
    cleaned_parts = []

    for p in parts:
        if p.startswith("(") and p.endswith(")"):
            cleaned_parts.append(p.upper())
        else:
            cleaned_parts.append(p.title())
    return " ".join(cleaned_parts)


In [ ]:
clean_birthplace("Pescia teramo")

In [ ]:
df['nato_a'] = df['nato_a'].apply(clean_birthplace)

In [ ]:
df['nato_a'].value_counts()

In [5]:
from vab_rt_import.utils.places_infer import build_place_inference

Il file CSV è già aggiornato.


In [ ]:
_worker = build_place_inference(col_name='nato_a', prefix='birth')
df = df.apply(_worker, axis=1)

In [ ]:
columns = ["nato_a", "birth_country", "birth_province", "birth_comune"]
df[columns]

## Fiscal data check

Qui dobbiamo avere le colonne pulite (o NA):
- birth_country
- birth_province
- birth_comune

In [ ]:
columns += ['nome_socio', 'cognome_socio', 'note']
columns = list(set(columns))

In [ ]:
df[columns][df['nato_il'].isna()]

In [6]:
from vab_rt_import.utils.fiscal_clean import safe_check_fiscal_data, cf_error_counts

In [ ]:
columns += ['cod_fisc', 'nato_il', '_cf_error', "nome_socio", "cognome_socio"]
columns = list(set(columns))

In [ ]:
result = df.apply(safe_check_fiscal_data, axis=1)
df_clean  = result[result['_cf_error'].isna()].drop(columns=['_cf_error'])
df_errors = result[result['_cf_error'].notna()]

print(f"OK:           {len(df_clean)}")
print(f"Con errori:   {len(df_errors)}")
print(f"\nContatori:\n{cf_error_counts}")

In [ ]:
df_errors[columns]

In [ ]:
df_errors['id_sezione_vab'].hist()

In [ ]:
df_errors[columns].to_csv('not_recoverable.csv')

In [ ]:
# remove invalid records
df = df_clean
df['birthdate'] = df['nato_il']
df['first_name'] = df['nome_socio']
df['last_name'] = df['cognome_socio']
df['gender'] = df['sesso_socio']
# drop unused columns
df.drop(columns=['nato_a', 'nato_il', 'nome_socio', 'cognome_socio'], inplace=True)

In [ ]:
df['cod_fisc']

# FISCAL DF

In [ ]:
FDF = dict()

In [ ]:
list(df.columns)

In [ ]:
columns = [
    'id',
    'first_name',
    'last_name',
    'gender',
    'birthdate',
    'birth_country',
    'birth_province',
    'birth_comune',
]

In [ ]:
df[columns]

In [ ]:
FDF['FISCAL'] = df[columns]

# Proceed

Ok, fiscal data has been processed, now we need to move forward.. Documents!

In [ ]:
list(df.columns)

In [ ]:
columns = [
    'patente_tipo',
    'patente_numero',
    'pat_rilasc_il',
    'pat_rilasc_da',
    'pat_scad',
    'tipo_doc_identita_socio',
    'numero_doc_identita_socio',
    'scadenza_doc_identita_socio',
    'doc_identita_socio_rilasc_da',
]

In [ ]:
df[columns]

In [ ]:
df['pat_rilasc_da']

In [ ]:
def clean_pat_rilasc_da(value):
    if is_empty(value):
        return pd.NA
    value = value.strip().upper()
    value = re.sub(r"\s+", "-", value)
    if len(value) <= 2:
        return pd.NA
    return value

df['pat_rilasc_da'].apply(clean_pat_rilasc_da)

In [ ]:
df['pat_rilasc_da'] = df['pat_rilasc_da'].apply(clean_pat_rilasc_da)

In [ ]:
def normalize_tipo_patente(value):
    if is_empty(value):
        return pd.NA
    value = str(value).strip().upper()
    value = re.sub(r'[\s\-\/,;.]+', '', value)

    found = set()
    i = 0
    while i < len(value):
        matched = False
        for token in ['C1E', 'D1E', 'AM', 'A1', 'A2', 'BE', 'B1', 'C1', 'CE', 'D1', 'DE', 'A', 'B', 'C', 'D', 'T']:
            if value[i:].startswith(token):
                found.add(token)
                i += len(token)
                matched = True
                break
        if not matched:
            i += 1
    if not found:
        return pd.NA
    return '-'.join(sorted(found))

In [ ]:
df['patente_tipo'].apply(normalize_tipo_patente).value_counts(dropna=False)

In [ ]:
df['patente_tipo'] = df['patente_tipo'].apply(normalize_tipo_patente)

In [ ]:
df['patente_numero']

In [ ]:
def clean_patente_number(value):
    if is_empty(value):
        return pd.NA

    value = str(value).strip().upper()
    value = re.sub(r"\s?", "", value)

    if len(value) != 10:
        return pd.NA

    return value

_t = df['patente_numero'].apply(clean_patente_number)
_t.isna().value_counts(dropna=False)
_t.apply(lambda x: len(str(x))).value_counts()

In [ ]:
df[df['patente_numero'].map(lambda x: len(str(x))) != 10]['patente_numero']

In [ ]:
df['patente_numero'] = df['patente_numero'].apply(clean_patente_number)

In [ ]:
df['patente_numero'].isna().value_counts()

In [ ]:
from pandas._libs.tslibs.parsing import DateParseError

def clean_data_rilascio(value):
    if is_empty(value):
        return pd.NaT
    try:
        value = pd.to_datetime(value)
    except DateParseError:
        return pd.NaT

    return value


In [ ]:
df['pat_rilasc_il'] = df['pat_rilasc_il'].apply(clean_data_rilascio)

In [ ]:
df['pat_scad'] = df['pat_scad'].apply(clean_data_rilascio)

### Documenti di identità

- 'tipo_doc_identita_socio',
- 'numero_doc_identita_socio',
- 'scadenza_doc_identita_socio',
- 'doc_identita_socio_rilasc_da',

In [ ]:
from rapidfuzz.distance import Levenshtein

CI_TYPE_MAP = {
    'CDI': [
        'ci', 'ca', 'cdi', 'cie', 'cid',
        'cartaidentita', 'cartadidentita', 'cartadiidentita',
        'cartadidentit', 'cartaidientita', 'catradiidentita',
        'cartadidentit', 'cidentita', 'cidentia', 'cidentit',
        'cidendita', 'cielettronica', 'cartaid', 'cartaceo',
        'carta', 'documentoidentita', 'docidnazionale',
        'tesseradidentita', 'cui',
    ],
    'CNS': [
        'tesserasanitaria', 'cns', 'tessera', 'codicefiscale',
    ],
    'PATENTE': [
        'patente', 'patenteauto', 'patentediguida', 'patenteguida',
        'patauto', 'patguida', 'pat', 'patentenautica',
        'patenteeuropea', 'patenteab', 'patenteade', 'patenteabc',
        'patentede', 'patentea',
    ],
    'PASSAPORTO': [
        'passaporto', 'passaportobritannico', 'pass',
        'psoggiorno',
    ],
    'ALTRO': [
        'tesseraministeriale', 'tesseramininterno',
        'ministerodellintern', 'cartamultiservizi',
        'permessosoggiorno', 'permessodisoggiorno', 'permdisoggiorno',
        'permessodisoggiorn', 'psoggiorno', 'cartasoggiorno',
        'portodarmi'
    ],
}


_DOC_JUNK = re.compile(r'^(x+|0+|\*+|aaa+|doc|cmd|comune|\d+)$')
def clean_tipo_doc_id(value):
    if is_empty(value):
        return pd.NA

    value = normalize_txt(str(value)).strip().lower()
    value = re.sub(r'[\s\'\.\-\_\/]+', '', value)

    if len(value) < 2 or _DOC_JUNK.match(value):
        return pd.NA

    for ctype, aliases in CI_TYPE_MAP.items():
        if value in aliases:
            return ctype

    if is_empty(value):
        return pd.NA


    prefix_map = {
        'pa': 'PATENTE',
        'pat':   'PATENTE',
        'pass':  'PASSAPORTO',
        'carta': 'CDI',
        'cid':   'CDI',
        'ci':    'CDI',
        'cod':   'CNS',
        'ca': 'CDI',
    }
    for prefix, ctype in prefix_map.items():
        if value.startswith(prefix):
            best_dist = min(
                Levenshtein.normalized_distance(value, alias)
                for alias in CI_TYPE_MAP[ctype]
            )
            if best_dist < 0.4:
                return ctype

    best_type, best_dist = None, 1.0
    for ctype, aliases in CI_TYPE_MAP.items():
        for alias in aliases:
            dist = Levenshtein.normalized_distance(value, alias)
            if dist < best_dist:
                best_dist = dist
                best_type = ctype

    if best_dist < 0.25:
        return best_type

    print(f"[tipo_doc] non riconosciuto: {value!r} (best={best_type!r}, dist={best_dist:.2f})")
    return pd.NA

df['_tipo_doc_identita_socio'] = df['tipo_doc_identita_socio'].apply(clean_tipo_doc_id)

In [ ]:
clean_tipo_doc_id('ci')

In [ ]:
df[['_tipo_doc_identita_socio','tipo_doc_identita_socio']]

In [ ]:
df['tipo_doc_identita_socio'] = df['_tipo_doc_identita_socio']
df.drop(columns=['_tipo_doc_identita_socio'], inplace=True)

In [ ]:
df['tipo_doc_identita_socio'].value_counts(dropna=False)

In [ ]:
df['scadenza_doc_identita_socio'] = (df['scadenza_doc_identita_socio'].apply
                                   (clean_data_rilascio))

In [ ]:
df['doc_identita_socio_rilasc_da']

In [ ]:
from id_docs_clean import clean_doc_id_rilasciato_da

In [ ]:
df['doc_identita_socio_rilasc_da'] = df['doc_identita_socio_rilasc_da'].apply(clean_doc_id_rilasciato_da)

# Residenza, indirizzi, email..

In [ ]:
list(df.columns)

In [ ]:
df['address_street'] = df['via']
df['address_comune'] = df['residente_a']
df['address_provincia'] = df['provincia']
df['address_cap'] = df['cap']

df['address_phone_home'] = df['tel_abit']
df['address_phone_mobile'] = df['tel_cellulare']
df['address_phone_other'] = df['tel_altro']

df.drop(columns=['via', 'residente_a', 'provincia', 'cap', 'tel_abit', 'tel_cellulare', 'tel_altro'], inplace=True)

In [ ]:
df

# Robe interne, puliamo via..

In [ ]:
# Clean columns
list(df.columns)

In [1]:
from vab_rt_import.utils import FiscalCountries
fc = FiscalCountries()

In [2]:
fc.get_name("Z301")

'ALGERIA'